## Project Instructions | Level Basic

* Create a pandas DataFrame called `nasdaq100_ca` containing the `nasdaq100_CA.csv` file and add a column called `"ytd"` containing year to date (YTD) performance for each company.
* Use the OpenAI API to classify each stock in the `nasdaq_ytd.csv` into a sector, saving as a new column in the `nasdaq` DataFrame called `"sector"` with the following values: Technology, Consumer Cyclical, Industrials, Utilities, Healthcare, Communication, Energy, Consumer Defensive, Real Estate, or Financial.
* Use the OpenAI API to provide summary information about the selected Nasdaq companies, recommending the two best sectors and two or more companies per sector, storing as a variable called `stock_recommendations`.

*Note that when clicking "Submit Project" it may take a while for the code to execute and the tests to run as you are interacting with the OpenAI API.

## Enriching stock market data using Open AI API 

The Nasdaq-100 is a stock market index made up of 101 equity securities issued by 100 of the largest non-financial companies listed on the Nasdaq stock exchange. It helps investors compare stock prices with previous prices to determine market performance.

In this project you are provided with two CSV files containing Nasdaq-100 stock information:
- _**nasdaq100_CA.csv**_: contains information about companies in the index such as symbol, name, etc. For this analysis, only companies headquartered in California have been selected.
- _**nasdaq100_price_change.csv**_: contains price changes per stock across periods including (but not limited to) one day, five days, one month, six months, one year, etc.

As an AI developer, you will leverage the OpenAI API to classify companies into sectors and produce a summary of sector and company performance for this year, for the companies in the index that are headquartered in California.

# CSV with Nasdaq-100 stock data

In this project, you have available two CSV files `nasdaq100_CA.csv` and `nasdaq100_price_change.csv`.

## nasdaq100_CA.csv

```py
symbol,name,headQuarter,dateFirstAdded,cik,founded
AAPL,Apple Inc.,"Cupertino, CA",,0000320193,1976-04-01
ABNB,Airbnb,"San Francisco, CA",,0001559720,2008-08-01
ADBE,Adobe Inc.,"San Jose, CA",,0000796343,1982-12-01
...
```

## nasdaq100_price_change.csv

```py
symbol,1D,5D,1M,3M,6M,ytd,1Y,3Y,5Y,10Y,max
AAPL,-1.7254,-8.30086,-6.20411,3.042,15.64824,42.99992,8.47941,60.96299,245.42031,976.99441,139245.53954
ABNB,2.1617,-2.21919,9.88336,19.43286,19.64241,68.66902,23.64013,-1.04347,-1.04347,-1.04347,-1.04347
ADBE,0.5409,-1.77817,9.16191,52.0465,38.01522,57.22723,21.96206,17.83037,109.05718,1024.69214,251030.66399
ADI,0.9291,-4.03352,2.58486,3.65887,5.01602,17.02062,8.09735,63.42847,92.81874,286.77518,26012.63736
...
```

In [ ]:
# ==========================================
# STEP 1: SETUP AND PREPARATION
# ==========================================

# Import the necessary toolkits (libraries)
import os             # Helps Python securely access environment variables (like passwords)
import pandas as pd   # The go-to tool for working with data tables (think of it as Python's Excel)
from openai import OpenAI # The tool that lets us talk to ChatGPT's brain

# Set up our connection to OpenAI
# We use an API key, which acts like a secret password to prove we have an account
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# ==========================================
# STEP 2: LOADING AND COMBINING THE DATA
# ==========================================

# Read our two data files into pandas DataFrames (which are just data tables)
nasdaq100_ca = pd.read_csv("nasdaq100_CA.csv")
price_change = pd.read_csv("nasdaq100_price_change.csv")

# Combine (merge) the two tables together. 
# We use the stock "symbol" (like AAPL for Apple) as the matching key. 
# This is very similar to doing a VLOOKUP in Excel to pull the "ytd" (Year-to-Date) performance into our main table.
nasdaq100_ca = nasdaq100_ca.merge(price_change[["symbol", "ytd"]], on="symbol", how="inner")

# Look at the first 5 rows of our newly combined table just to make sure it looks right
nasdaq100_ca.head()

# ==========================================
# STEP 3: ASKING AI TO CLASSIFY EACH STOCK
# ==========================================

# Start a loop: We are going to look at every single stock symbol in our table, one by one.
for company in nasdaq100_ca["symbol"]:
    
    # Write the specific instruction (prompt) we want to send to the AI for this specific company.
    prompt = f'''Classify company {company} into one of the following sectors. Answer only with the sector name: Technology, Consumer Cyclical, Industrials, Utilities, Healthcare, Communication, Energy, Consumer Defensive, Real Estate, Financial.'''
    
    # Send the instruction to the OpenAI model (gpt-3.5-turbo)
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0, # Temperature 0.0 means we want strict, factual answers, not creative ones
    )
    
    # Dig into the AI's response to extract just the text it replied with
    sector = response.choices[0].message.content
    
    # Find the row in our table where the symbol matches the current company, 
    # and save the AI's answer into a brand new column called "Sector"
    nasdaq100_ca.loc[nasdaq100_ca["symbol"] == company, "Sector"] = sector
    
# Tally up the results to see how many companies ended up in each sector
nasdaq100_ca["Sector"].value_counts()

# ==========================================
# STEP 4: GETTING AI RECOMMENDATIONS
# ==========================================

# Now that our table has both 'ytd' performance and 'Sector' categories, 
# let's feed the whole table back to the AI and ask for investment recommendations.
prompt = f'''Provide summary information about Nasdaq-100 stock performance year to date (YTD) of companies headquartered in CA, recommending the two best sectors and two or more companies per sector.
            Company data: {nasdaq100_ca} 
'''

# Send this final request to the AI
response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
    )

# Extract the AI's final written report
stock_recommendations = response.choices[0].message.content

# Print the report to the screen so we can read it!
print(stock_recommendations)